# SpatialPeeler benchmark — Slide-seq Puck 06

Runs SpatialPeeler on the generated benchmark data from `benchmark.ipynb`.

**Setup:** Puck 06 bottom half split into case / control.  
Case = same tissue, but expression of the top 100 PC2-loaded genes is doubled (Poisson-perturbed) inside a central circle (~10% of beads).  
Control = the unmodified bottom section.

**Ground truth:** `obs['in_circle']` — beads that were perturbed in the case sample.

**Goal:** verify that SpatialPeeler recovers a factor whose p_hat score is elevated specifically inside the perturbed circle.

In [ ]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import scanpy as sc
import anndata as ad
from sklearn.decomposition import NMF
from sklearn.exceptions import ConvergenceWarning
from sklearn.metrics import roc_auc_score, roc_curve

warnings.simplefilter('ignore', category=ConvergenceWarning)
sc.settings.verbosity = 1

# SpatialPeeler imports
ROOT = Path('/lustre/scratch126/gengen/teams_v2/marks/dp31/SpatialPeeler')
sys.path.insert(0, str(ROOT))
from SpatialPeeler import case_prediction as cpred
from SpatialPeeler import plotting as plot

RAND_SEED = 28
np.random.seed(RAND_SEED)

Updated sys.path: ['/lustre/scratch126/gengen/teams_v2/marks/dp31', '/lustre/scratch126/gengen/teams_v2/marks/dp31', '/lustre/scratch126/gengen/teams_v2/marks/dp31/SpatialPeeler', '/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/lustre/scratch126/gengen/teams_v2/marks/dp31/SpatialPeeler/.venv/lib/python3.10/site-packages']
Updated sys.path: ['/lustre/scratch126/gengen/teams_v2/marks/dp31', '/lustre/scratch126/gengen/teams_v2/marks/dp31', '/lustre/scratch126/gengen/teams_v2/marks/dp31', '/lustre/scratch126/gengen/teams_v2/marks/dp31/SpatialPeeler', '/usr/lib/python310.zip', '/usr/lib/python3.10', '/usr/lib/python3.10/lib-dynload', '', '/lustre/scratch126/gengen/teams_v2/marks/dp31/SpatialPeeler/.venv/lib/python3.10/site-packages']


## 1. Load benchmark data

In [ ]:
DATA_DIR = ROOT / 'benchmark' / 'generated_benchmark_data'

adata_ctrl = ad.read_h5ad(DATA_DIR/'adata06_top.h5ad')
adata_case = ad.read_h5ad(DATA_DIR/'adata06_bot_case.h5ad')

print('Control:', adata_ctrl)
print('Case:   ', adata_case)

In [ ]:
# Assign case/control labels
adata_ctrl.obs['sample_id'] = 'puck06_bot_ctrl'
adata_ctrl.obs['status']    = 0
adata_ctrl.obs['Condition'] = 'Control'

adata_case.obs['sample_id'] = 'puck06_bot_case'
adata_case.obs['status']    = 1
adata_case.obs['Condition'] = 'Case'

# Confirm ground truth column is present in both
assert 'in_circle' in adata_ctrl.obs.columns, "'in_circle' missing from control obs"
assert 'in_circle' in adata_case.obs.columns, "'in_circle' missing from case obs"

print(f"Control in_circle: {adata_ctrl.obs['in_circle'].sum():,} / {adata_ctrl.n_obs:,}")
print(f"Case    in_circle: {adata_case.obs['in_circle'].sum():,} / {adata_case.n_obs:,}")

## 2. Concatenate and preprocess

In [ ]:
adata = ad.concat(
    [adata_ctrl, adata_case],
    join='inner',
    label='sample_id',
    keys=['puck06_bot_ctrl', 'puck06_bot_case'],
    index_unique='-'
)

# Restore obs columns overwritten by concat (concat only keeps obs cols from the slices)
obs_ctrl = adata_ctrl.obs[['sample_id', 'status', 'Condition', 'in_circle']].copy()
obs_case = adata_case.obs[['sample_id', 'status', 'Condition', 'in_circle']].copy()
obs_ctrl.index = obs_ctrl.index + '-0'
obs_case.index = obs_case.index + '-1'
obs_merged = pd.concat([obs_ctrl, obs_case])

for col in ['sample_id', 'status', 'Condition', 'in_circle']:
    adata.obs[col] = obs_merged.loc[adata.obs_names, col].values

adata.obs['status'] = adata.obs['status'].astype(int)
adata.obs['in_circle'] = adata.obs['in_circle'].astype(bool)

print(adata)
print(adata.obs[['sample_id', 'status', 'Condition', 'in_circle']].value_counts())

In [ ]:
# Store raw counts before normalisation
adata.layers['counts'] = adata.X.copy()

# Filter lowly expressed genes and low-count spots
min_cells = max(1, adata.n_obs // 500)
sc.pp.filter_genes(adata, min_cells=min_cells)

adata.obs['total_counts'] = np.array(adata.X.sum(axis=1)).flatten()
sc.pp.filter_cells(adata, min_counts=100)

print(f'After filtering: {adata.shape}')

In [ ]:
# HVG selection (batch-aware per sample)
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    batch_key='sample_id',
    flavor='seurat_v3',
    layer='counts',
    subset=True
)
print(f'HVG-filtered: {adata.shape}')

## 3. NMF decomposition

In [ ]:
N_FACTORS = 20

# Use raw counts for NMF (non-negative)
X_nmf = adata.layers['counts']
if sp.issparse(X_nmf):
    X_nmf = X_nmf.toarray()
X_nmf = X_nmf.astype(np.float32)

nmf_model = NMF(
    n_components=N_FACTORS,
    init='nndsvda',
    random_state=RAND_SEED,
    max_iter=1000,
    solver='cd',
)
W = nmf_model.fit_transform(X_nmf)   # (spots × factors)
H = nmf_model.components_            # (factors × genes)

adata.obsm['X_nmf'] = W
adata.uns['nmf_components'] = H

print(f'NMF W shape: {W.shape}')
print(f'Reconstruction error: {nmf_model.reconstruction_err_:.2f}')

In [ ]:
# Quick spatial check: plot a few NMF factors for each sample
sample_ids = adata.obs['sample_id'].unique().tolist()
adata_by_sample = {
    sid: adata[adata.obs['sample_id'] == sid].copy()
    for sid in sample_ids
}

for factor_idx in range(min(5, N_FACTORS)):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, sid in zip(axes, sample_ids):
        sub = adata_by_sample[sid]
        x = sub.obs['Raw_Slideseq_X'].values.astype(float)
        y = sub.obs['Raw_Slideseq_Y'].values.astype(float)
        scores = sub.obsm['X_nmf'][:, factor_idx]
        vmax = np.percentile(scores, 99)
        sc_ = ax.scatter(x, y, c=scores, s=0.3, cmap='viridis',
                         vmin=0, vmax=vmax, linewidths=0, rasterized=True)
        plt.colorbar(sc_, ax=ax, fraction=0.03, pad=0.02)
        ax.set_aspect('equal', 'box')
        ax.axis('off')
        ax.set_title(f'Factor {factor_idx+1} — {sid}', fontsize=9)
    plt.tight_layout()
    plt.show()

## 4. Run SpatialPeeler

In [ ]:
results = cpred.single_factor_logistic_evaluation(
    adata,
    factor_key='X_nmf',
    max_factors=N_FACTORS
)

In [ ]:
coef_df = pd.DataFrame({
    'factor':     [f'Factor_{r["factor_index"]+1}' for r in results],
    'factor_idx': [r['factor_index'] for r in results],
    'coef':       [r['coef'] for r in results],
    'pval':       [r['pval'] for r in results],
}).sort_values('coef', ascending=False)

print(coef_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#e84545' if c > 0 else '#4575b4' for c in coef_df['coef']]
ax.bar(coef_df['factor'], coef_df['coef'], color=colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xlabel('Factor')
ax.set_ylabel('Logistic regression coefficient')
ax.set_title('SpatialPeeler: factor coefficients (case vs control)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Spatial visualisation of p_hat

In [ ]:
# Pick the top GOF (positive coef) factors to inspect spatially
top_factors = coef_df[coef_df['coef'] > 0].head(5)['factor_idx'].tolist()
print('Top GOF factors:', [f+1 for f in top_factors])

for fi in top_factors:
    p_hat = results[fi]['p_hat']
    adata.obs['p_hat'] = p_hat.astype('float32')

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, sid in zip(axes, sample_ids):
        sub = adata_by_sample[sid]
        sub.obs['p_hat'] = adata.obs.loc[sub.obs_names, 'p_hat'].values

        x = sub.obs['Raw_Slideseq_X'].values.astype(float)
        y = sub.obs['Raw_Slideseq_Y'].values.astype(float)
        ph = sub.obs['p_hat'].values

        sc_ = ax.scatter(x, y, c=ph, s=0.4, cmap='magma_r',
                         vmin=0, vmax=1, linewidths=0, rasterized=True)
        plt.colorbar(sc_, ax=ax, fraction=0.03, pad=0.02, label='p_hat')

        # Overlay circle boundary on the case sample
        if sid == 'puck06_bot_case':
            circle_mask = sub.obs['in_circle'].values
            cx = x[circle_mask].mean()
            cy = y[circle_mask].mean()
            r  = np.sqrt(circle_mask.sum() / np.pi)  # approximate radius from bead count
            ax.add_patch(plt.Circle((cx, cy), r * 1.5, fill=False,
                                    edgecolor='cyan', linewidth=1.2,
                                    label='perturbed circle'))
            ax.legend(fontsize=8, frameon=False)

        ax.set_aspect('equal', 'box')
        ax.axis('off')
        ax.set_title(f'Factor {fi+1} p_hat — {sid}', fontsize=9)

    plt.suptitle(f'Factor {fi+1}  (coef={coef_df.loc[coef_df["factor_idx"]==fi, "coef"].values[0]:.3f})',
                 fontsize=11)
    plt.tight_layout()
    plt.show()

## 6. Evaluate recovery of the perturbed circle

For each factor, compute AUROC between p_hat (on case beads only) and the ground-truth `in_circle` label.  
A high AUROC means the factor's p_hat distinguishes perturbed from unperturbed case beads.

In [ ]:
case_mask = adata.obs['status'].values == 1
gt        = adata.obs['in_circle'].values[case_mask].astype(int)

auroc_rows = []
for r in results:
    ph = r['p_hat'][case_mask]
    try:
        auc = roc_auc_score(gt, ph)
    except Exception:
        auc = np.nan
    auroc_rows.append({'factor': f'Factor_{r["factor_index"]+1}',
                       'factor_idx': r['factor_index'],
                       'coef': r['coef'],
                       'auroc': auc})

auroc_df = pd.DataFrame(auroc_rows).sort_values('auroc', ascending=False)
print(auroc_df.to_string(index=False))

In [ ]:
# Bar chart of AUROC per factor
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(auroc_df['factor'], auroc_df['auroc'],
       color=['#e84545' if a >= 0.7 else '#aaaaaa' for a in auroc_df['auroc']])
ax.axhline(0.5, color='black', linewidth=0.8, linestyle='--', label='random')
ax.set_ylim(0, 1)
ax.set_xlabel('Factor')
ax.set_ylabel('AUROC (p_hat vs in_circle, case beads only)')
ax.set_title('Circle recovery per factor')
ax.legend(fontsize=9)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ROC curve for the best factor
best_fi   = auroc_df.iloc[0]['factor_idx']
best_auc  = auroc_df.iloc[0]['auroc']
best_ph   = results[int(best_fi)]['p_hat'][case_mask]

fpr, tpr, _ = roc_curve(gt, best_ph)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, color='#e84545', lw=2,
        label=f'Factor {int(best_fi)+1}  (AUC={best_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k--', lw=0.8)
ax.set_xlabel('False positive rate')
ax.set_ylabel('True positive rate')
ax.set_title('ROC — best factor (p_hat vs in_circle, case beads)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Spatial overlay: p_hat coloured by in_circle ground truth
sub_case = adata_by_sample['puck06_bot_case']
sub_case.obs['p_hat']     = adata.obs.loc[sub_case.obs_names, 'p_hat'].values
sub_case.obs['in_circle'] = adata.obs.loc[sub_case.obs_names, 'in_circle'].values

x  = sub_case.obs['Raw_Slideseq_X'].values.astype(float)
y  = sub_case.obs['Raw_Slideseq_Y'].values.astype(float)
ph = results[int(best_fi)]['p_hat'][case_mask]
gt_bool = sub_case.obs['in_circle'].values.astype(bool)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: p_hat
sc_ = axes[0].scatter(x, y, c=ph, s=0.5, cmap='magma_r',
                      vmin=0, vmax=1, linewidths=0, rasterized=True)
plt.colorbar(sc_, ax=axes[0], fraction=0.03, pad=0.02, label='p_hat')
axes[0].set_title(f'Case — Factor {int(best_fi)+1} p_hat', fontsize=10)

# Right: ground truth
colors = np.where(gt_bool, '#e84545', '#d3d3d3')
axes[1].scatter(x, y, c=colors, s=0.5, linewidths=0, rasterized=True)
patches = [mpatches.Patch(color='#e84545', label='in circle (perturbed)'),
           mpatches.Patch(color='#d3d3d3', label='outside circle')]
axes[1].legend(handles=patches, fontsize=8, frameon=False, loc='lower right')
axes[1].set_title('Case — ground truth (in_circle)', fontsize=10)

for ax in axes:
    ax.set_aspect('equal', 'box')
    ax.axis('off')

plt.suptitle(f'Best factor: Factor {int(best_fi)+1}  (AUROC={best_auc:.3f})', fontsize=11)
plt.tight_layout()
plt.show()